# 导入必要的库
导入Python标准库和常用数据科学库。

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras import regularizers
import random
import time

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\s

## 定义类别编码函数
将类别字符串编码为整数，并返回编码后的DataFrame和类别映射字典。

In [2]:
def encode_class_column(df, class_column='ClassName', new_column='num'):
    """
    将类别字符串编码为整数，返回新DataFrame和类别映射字典
    """
    uni = df[class_column].unique()
    mapping = {item: i for i, item in enumerate(uni)}
    df[new_column] = df[class_column].map(mapping)
    return df, mapping

## 评估相关函数
包含GPU设置、数据处理、模型构建、损失函数、训练与多次评估等。

In [3]:
def set_seed(seed=42):
    tf.keras.backend.clear_session()
    random.seed(seed)
    np.random.seed(seed)
    tf.set_random_seed(seed)

def setup_gpu():
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"检测到{len(gpus)}块GPU，已设置显存按需分配。")
        except RuntimeError as e:
            print(e)
    else:
        print("未检测到GPU，使用CPU运行。")

def split_data(data, test_ratio=0.2):
    idx = np.random.permutation(len(data))
    data_all = data.iloc[idx, :]
    split_idx = int(len(data_all) * (1 - test_ratio))
    train = data_all.iloc[:split_idx]
    test = data_all.iloc[split_idx:]
    return train, test

def get_xy(df):
    # 只保留除 className 和 num 以外的特征列
    feature_cols = [col for col in df.columns if col not in ['className', 'ClassName', 'num']]
    x = df[feature_cols]
    y = df['num'].values
    return x, y

def get_class_map(data):
    return data.drop_duplicates(subset='num').set_index('num')['ClassName'].to_dict()

def build_model(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(512, activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def multi_category_focal_loss2(gamma=2., alpha=.25, class_weights=None):
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    if class_weights is not None:
        weights = np.array([class_weights.get(i, 1.0) for i in range(len(class_weights))])
        class_weights_tf = tf.constant(weights, dtype=tf.float32)
    else:
        class_weights_tf = None
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        y_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(1. - y_t, gamma)
        fl = alpha_t * weight * ce
        if class_weights_tf is not None:
            fl = fl * class_weights_tf
        return tf.reduce_mean(fl)
    return focal_loss_fixed

def train_and_evaluate(train_df, test_df, class_map, verbose=0):
    x_train, y_train = get_xy(train_df)
    x_test, y_test = get_xy(test_df)
    num_classes = len(class_map)
    # 转成one-hot编码
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
    class_weights = dict(enumerate(class_weights_array))
    model = build_model(x_train.shape[1], num_classes)
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(loss=multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights),
                  optimizer=optimizer, metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
    model.fit(x_train, y_train_cat, epochs=500, batch_size=64, verbose=verbose, callbacks=[reduce_lr, early_stopping])
    y_pred = model.predict(x_test).argmax(axis=1)
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_test, y_pred, labels=list(class_map.keys()))
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    return [acc, precision, recall, f1], per_class_acc

def multi_run_eval(data, mapping_df, n_runs=10, test_ratio=0.2, seed=42):
    set_seed(seed)
    class_map = get_class_map(data)
    all_class_ids = sorted(class_map.keys())
    class_labels = [class_map[i] for i in all_class_ids]
    num2veg = dict(zip(mapping_df['num'], mapping_df['Veg_Formation']))
    overall_scores = []
    per_class_accs = []
    overall_veg_scores = []
    for run in range(n_runs):
        print(f"\n===== 正在进行第 {run+1}/{n_runs} 次训练与评估 =====")
        train_df, test_df = split_data(data, test_ratio)
        x_train, y_train = get_xy(train_df)
        x_test, y_test = get_xy(test_df)
        num_classes = len(class_map)
        y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
        class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
        class_weights = dict(enumerate(class_weights_array))
        model = build_model(x_train.shape[1], num_classes)
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
        model.compile(loss=multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights),
                      optimizer=optimizer, metrics=['accuracy'])
        early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
        model.fit(x_train, y_train_cat, epochs=500, batch_size=64, verbose=0, callbacks=[reduce_lr, early_stopping])
        y_pred_num = model.predict(x_test).argmax(axis=1)
        # 细类四项指标
        acc = accuracy_score(y_test, y_pred_num)
        precision = precision_score(y_test, y_pred_num, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred_num, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred_num, average='macro', zero_division=0)
        cm = confusion_matrix(y_test, y_pred_num, labels=list(class_map.keys()))
        per_class_acc = cm.diagonal() / cm.sum(axis=1)
        # 大类指标
        y_true_veg = [num2veg[n] for n in y_test]
        y_pred_veg = [num2veg[n] for n in y_pred_num]
        acc_veg = accuracy_score(y_true_veg, y_pred_veg)
        precision_veg = precision_score(y_true_veg, y_pred_veg, average='macro', zero_division=0)
        recall_veg = recall_score(y_true_veg, y_pred_veg, average='macro', zero_division=0)
        f1_veg = f1_score(y_true_veg, y_pred_veg, average='macro', zero_division=0)
        overall_veg_scores.append([acc_veg, precision_veg, recall_veg, f1_veg])
        if len(per_class_acc) < len(all_class_ids):
            acc_full = np.full(len(all_class_ids), np.nan)
            acc_full[:len(per_class_acc)] = per_class_acc
            per_class_accs.append(acc_full)
        else:
            per_class_accs.append(per_class_acc)
        overall_scores.append([acc, precision, recall, f1])
        print(f"Run {run+1}: acc={acc:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")
        print(f"Veg_Formation: acc={acc_veg:.4f}, precision={precision_veg:.4f}, recall={recall_veg:.4f}, f1={f1_veg:.4f}")
    # 汇总表格
    score_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    veg_score_names = ['Veg_Formation_Accuracy', 'Veg_Formation_Precision', 'Veg_Formation_Recall', 'Veg_Formation_F1']
    index = score_names + veg_score_names + class_labels
    results = np.vstack([
        np.array(overall_scores).T,
        np.array(overall_veg_scores).T,
        np.array(per_class_accs).T
    ])
    df_results = pd.DataFrame(results, index=index, columns=[f'Run_{i+1}' for i in range(n_runs)])
    df_results['Mean'] = df_results.mean(axis=1)
    print('\n多次整体与各类别准确率统计表：')
    print(df_results)
    return df_results

## 定义主流程函数
包含数据加载、类别编码、模型训练与评估等完整流程。

In [4]:
def main(code_csv_path, mapping_csv_path, result_csv_path, n_runs=10, test_ratio=0.2, seed=42):
    """
    读取原始特征文件和类别映射表，将类别映射为num后评估，并统计大类指标
    """
    # 读取原始特征数据
    df = pd.read_csv(code_csv_path, encoding='utf_8_sig')
    # 读取类别映射表
    mapping_df = pd.read_csv(mapping_csv_path, encoding='utf_8_sig')
    mapping = dict(zip(mapping_df['ClassName'], mapping_df['num']))
    # 添加num列
    df['num'] = df['ClassName'].map(mapping)
    # 检查是否有未映射的类别
    if df['num'].isnull().any():
        raise ValueError('有ClassName未能映射到num，请检查类别映射表！')
    # 多次评估，传入mapping_df用于大类统计
    df_results = multi_run_eval(df, mapping_df, n_runs=n_runs, test_ratio=test_ratio, seed=seed)
    # 保存评估结果
    df_results.to_csv(result_csv_path, encoding='utf_8_sig')
    # 生成数字到类别的反向映射
    num2class = dict(zip(mapping_df['num'], mapping_df['ClassName']))
    return num2class, df_results

## 调用主流程函数并输出结果
请根据实际数据路径修改参数。

In [5]:
# 示例：请根据实际路径修改
code_csv = 'F:/TensorFlow/xinjiang/traindata20250626_2.csv'
mapping_csv = 'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv'
result_csv = 'F:/TensorFlow/xinjiang/XJruns20250627_50.csv'

num2class, df_results = main(code_csv, mapping_csv, result_csv, n_runs=50, test_ratio=0.2, seed=42)
print('数字到类别映射:', num2class)
display(df_results)



===== 正在进行第 1/50 次训练与评估 =====
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor

===== 正在进行第 1/50 次训练与评估 =====
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Run 1: acc=0.4086, precision=0.3023, recall=0.3515, f1=0.3094
Veg_Formation: acc=0.5735, precision=0.4285, recall=0.4141, f1=0.4180

===== 正在进行第 2/50 次训练与评估 =====
Run 1: acc=0.4086, precision=0.3023, recall=0.3515, f1=0.3094
Veg_Formation: acc=0.5735, precision=0.4285, recall=0.4141, f1=0.4180

===== 正在进行第 2/50 次训练与评估 =====


d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:130: RuntimeWarning: invalid value encountered in true_divide


Run 2: acc=0.4588, precision=0.2889, recall=0.2941, f1=0.2710
Veg_Formation: acc=0.6093, precision=0.4535, recall=0.5050, f1=0.4601

===== 正在进行第 3/50 次训练与评估 =====
Run 3: acc=0.4480, precision=0.2967, recall=0.3231, f1=0.2886
Veg_Formation: acc=0.6237, precision=0.5521, recall=0.5987, f1=0.5479

===== 正在进行第 4/50 次训练与评估 =====
Run 3: acc=0.4480, precision=0.2967, recall=0.3231, f1=0.2886
Veg_Formation: acc=0.6237, precision=0.5521, recall=0.5987, f1=0.5479

===== 正在进行第 4/50 次训练与评估 =====


d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:130: RuntimeWarning: invalid value encountered in true_divide


Run 4: acc=0.4803, precision=0.3669, recall=0.4425, f1=0.3617
Veg_Formation: acc=0.6201, precision=0.5819, recall=0.5821, f1=0.5250

===== 正在进行第 5/50 次训练与评估 =====


...

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:130: RuntimeWarning: invalid value encountered in true_divide


Run 46: acc=0.5054, precision=0.4309, recall=0.3991, f1=0.3807
Veg_Formation: acc=0.6308, precision=0.4772, recall=0.5150, f1=0.4842

===== 正在进行第 47/50 次训练与评估 =====


d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:130: RuntimeWarning: invalid value encountered in true_divide


Run 47: acc=0.4122, precision=0.2993, recall=0.3188, f1=0.2843
Veg_Formation: acc=0.5520, precision=0.4799, recall=0.4594, f1=0.4596

===== 正在进行第 48/50 次训练与评估 =====
Run 48: acc=0.4122, precision=0.3016, recall=0.3261, f1=0.2950
Veg_Formation: acc=0.5735, precision=0.4525, recall=0.4344, f1=0.4402

===== 正在进行第 49/50 次训练与评估 =====
Run 48: acc=0.4122, precision=0.3016, recall=0.3261, f1=0.2950
Veg_Formation: acc=0.5735, precision=0.4525, recall=0.4344, f1=0.4402

===== 正在进行第 49/50 次训练与评估 =====
Run 49: acc=0.4695, precision=0.3500, recall=0.4317, f1=0.3548
Veg_Formation: acc=0.6272, precision=0.5031, recall=0.6092, f1=0.5190

===== 正在进行第 50/50 次训练与评估 =====
Run 49: acc=0.4695, precision=0.3500, recall=0.4317, f1=0.3548
Veg_Formation: acc=0.6272, precision=0.5031, recall=0.6092, f1=0.5190

===== 正在进行第 50/50 次训练与评估 =====
Run 50: acc=0.4265, precision=0.3111, recall=0.3119, f1=0.2955
Veg_Formation: acc=0.6201, precision=0.4709, recall=0.4745, f1=0.4705

多次整体与各类别准确率统计表：
                         

,Run_1,Run_2,Run_3,Run_4,Run_5,Run_6,Run_7,Run_8,Run_9,Run_10,...,Run_42,Run_43,Run_44,Run_45,Run_46,Run_47,Run_48,Run_49,Run_50,Mean
Accuracy,0.408602,0.458781,0.448029,0.480287,0.437276,0.419355,0.430108,0.419355,0.419355,0.401434,...,0.433692,0.455197,0.415771,0.354839,0.505376,0.412186,0.412186,0.469534,0.426523,0.433333
Precision,0.302297,0.288890,0.296706,0.366897,0.306211,0.278158,0.321136,0.286229,0.359221,0.263140,...,0.224341,0.341178,0.298540,0.258109,0.430940,0.299257,0.301623,0.349957,0.311142,0.313613
Recall,0.351516,0.294074,0.323148,0.442535,0.371425,0.301409,0.319830,0.326312,0.378292,0.301206,...,0.277350,0.342369,0.334657,0.288727,0.399124,0.318802,0.326086,0.431745,0.311867,0.347995
F1 Score,0.309420,0.271016,0.288636,0.361735,0.299736,0.270286,0.285573,0.284826,0.336739,0.265594,...,0.231011,0.312162,0.287622,0.251396,0.380726,0.284343,0.295017,0.354769,0.295527,0.302850
Veg_Formation_Accuracy,0.573477,0.609319,0.623656,0.620072,0.594982,0.587814,0.591398,0.605735,0.609319,0.569892,...,0.584229,0.648746,0.605735,0.523297,0.630824,0.551971,0.573477,0.627240,0.620072,0.600358
Veg_Formation_Precision,0.428505,0.453466,0.552119,0.581868,0.547106,0.465877,0.497937,0.470221,0.566248,0.416422,...,0.482250,0.570331,0.491796,0.395739,0.477249,0.479888,0.452505,0.503125,0.470921,0.498031
Veg_Formation_Recall,0.414140,0.505035,0.598692,0.582150,0.551436,0.491308,0.450248,0.468066,0.575252,0.415550,...,0.477146,0.621885,0.555667,0.415311,0.514986,0.459448,0.434384,0.609216,0.474498,0.510767
Veg_Formation_F1,0.417974,0.460086,0.547912,0.524997,0.522908,0.464942,0.462134,0.464748,0.554828,0.411995,...,0.470773,0.549956,0.502705,0.385889,0.484176,0.459631,0.440236,0.518986,0.470521,0.485928
针茅属荒漠草原,0.052632,0.166667,0.000000,0.083333,0.384615,0.058824,0.181818,0.176471,0.181818,0.000000,...,0.166667,0.066667,0.230769,0.058824,0.066667,0.250000,0.200000,0.187500,0.153846,0.124547
紫花针茅草原,0.833333,0.300000,0.444444,0.428571,0.444444,0.400000,0.692308,0.307692,0.333333,0.625000,...,0.500000,0.090909,0.583333,0.333333,0.916667,0.272727,0.571429,0.333333,0.714286,0.492223
